# 04 · Identification des variantes du dialecte tunisien (extension 2)

**Objectifs**
1. Vérifier quelles informations du corpus pourraient servir d'**annotations dialectales**.
2. Définir des classes en fonction de ce qui existe **réellement**, et en discuter la validité.
3. Entraîner des classifieurs à partir du **texte** et de l'**audio**, avec un protocole sans fuite (par talk).
4. Évaluer (accuracy, précision, rappel, F1), analyser les confusions et les limites.

**Conclusion anticipée de l'audit (section 2)** : TEDxTN ne contient **aucune annotation dialectale ni
régionale** par segment ou par locuteur. Nous construisons donc des étiquettes **indirectes** (région de
l'événement TEDx) en le signalant clairement, et nous ajoutons une tâche voisine aux étiquettes fiables :
reconnaître depuis l'audio le **degré de code-switching** du segment (TUN / MIXED / FOR).

## 1. Installation

In [ ]:
import os, subprocess, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/roudayn-kouka/tedxtn-tunisian-speech.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("TEDXTN_WORKDIR", "/content/drive/MyDrive/tedxtn")
    project = Path("/content/drive/MyDrive/tedxtn-tunisian-speech")
    if not project.exists():
        project = Path("/content/tedxtn-tunisian-speech")
        if not project.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(project)], check=True)
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from common import data, text
from common.config import RESULTS_DIR, SEED, SMOKE, env_flag
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
try:
    import torch; HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
RUN_AUDIO = env_flag("AUDIO", HAS_GPU)   # embeddings w2v-BERT-2.0 (inférence fp16, ~1,2 Go de VRAM)

df = data.load_corpus(download=os.environ.get("TEDXTN_SKIP_DOWNLOAD") != "1")

## 2. Audit des annotations disponibles

In [ ]:
print("Colonnes des CSV officiels :", ["audio_filename", "start_time", "end_time", "transcription", "translation"])
print("Talks :", df.talk.nunique(), "| événements TEDx :", df.event.nunique())
display(df.groupby("event").agg(talks=("talk", "nunique"), segments=("seg_id", "size")).sort_values("talks", ascending=False))

Aucune colonne ne décrit la région, l'origine du locuteur ou la variante dialectale. Le papier précise
seulement que les talks viennent de **11 villes** et donne le genre des locuteurs au niveau du corpus (pas par
segment). La seule information géographique exploitable est le **nom de l'événement TEDx** (TEDxNabeul,
TEDxPiloteAriana...), qui indique la ville où le talk a été donné.

**Hypothèse (forte) et ses limites** : la ville de l'événement est un indicateur bruité de la variante parlée.
Un orateur de Sfax peut parler à un TEDx de Tunis ; un événement universitaire réunit des étudiants de tout le
pays. Les résultats ci-dessous mesurent donc la capacité à reconnaître la **région de l'événement**, pas une
variante dialectale certifiée.

In [ ]:
REGION_KEYS = [  # (motif dans le nom de l'événement, région) — l'ordre compte : motifs longs d'abord
    ("enisousse", "Sahel"), ("eniso", "Sahel"), ("enim", "Sahel"), ("enis", "Sud"), ("enig", "Sud"), ("enib", "Nord"),
    ("tunis", "Grand Tunis"), ("carthage", "Grand Tunis"), ("ariana", "Grand Tunis"), ("manouba", "Grand Tunis"),
    ("benarous", "Grand Tunis"), ("marsa", "Grand Tunis"), ("bardo", "Grand Tunis"),
    ("menzah", "Grand Tunis"), ("ennasr", "Grand Tunis"), ("insat", "Grand Tunis"), ("enit", "Grand Tunis"),
    ("ensi", "Grand Tunis"), ("esprit", "Grand Tunis"), ("supcom", "Grand Tunis"), ("ihec", "Grand Tunis"),
    ("medtech", "Grand Tunis"), ("essec", "Grand Tunis"), ("fst", "Grand Tunis"),
    ("nabeul", "Cap Bon"), ("hammamet", "Cap Bon"), ("kelibia", "Cap Bon"), ("korba", "Cap Bon"),
    ("bizerte", "Nord"), ("beja", "Nord"), ("jendouba", "Nord"), ("kef", "Nord"), ("tabarka", "Nord"), ("zaghouan", "Nord"), ("siliana", "Nord"),
    ("sousse", "Sahel"), ("monastir", "Sahel"), ("mahdia", "Sahel"), ("moknine", "Sahel"), ("ksarhellal", "Sahel"),
    ("kairouan", "Centre"), ("kasserine", "Centre"), ("sidibouzid", "Centre"), ("gafsa", "Centre"),
    ("sfax", "Sud"), ("gabes", "Sud"), ("djerba", "Sud"), ("medenine", "Sud"), ("tataouine", "Sud"), ("tozeur", "Sud"),
    ("kebili", "Sud"), ("zarzis", "Sud"),
]
MANUAL_EVENT_REGION = {
    # À compléter par l'équipe après vérification (site de l'événement, description YouTube) :
    # "TEDxAPBS": "Grand Tunis",
}

def event_region(ev):
    if ev in MANUAL_EVENT_REGION:
        return MANUAL_EVENT_REGION[ev]
    e = ev.lower().replace("tedx", "")
    for k, r in REGION_KEYS:
        if k in e:
            return r
    return None

df["region"] = df.event.map(event_region)
unmapped = df[df.region.isna()].groupby("event").talk.nunique()
print("Événements non attribués (à compléter dans MANUAL_EVENT_REGION) :", dict(unmapped))
talks_per_region = df.dropna(subset=["region"]).groupby("region").talk.nunique().sort_values(ascending=False)
display(talks_per_region.rename("talks"))

## 3. Définition des classes
Une région n'est gardée que si elle compte **au moins 4 talks** : en dessous, aucune validation croisée par
talk n'est possible et le classifieur reconnaîtrait des **locuteurs**, pas des variantes. Les autres sont
regroupées ou écartées.

In [ ]:
MIN_TALKS = 4
keep = talks_per_region[talks_per_region >= MIN_TALKS].index.tolist()
D = df[df.region.isin(keep) & (df.n_words >= 3)].reset_index(drop=True)
print("Classes retenues :", keep)
display(D.groupby("region").agg(talks=("talk", "nunique"), segments=("seg_id", "size")))
FEASIBLE = len(keep) >= 2
if not FEASIBLE:
    print("Moins de deux régions exploitables : la tâche régionale n'est pas réalisable avec ce corpus (voir section 6).")

## 4. Classifieur textuel
TF-IDF sur n-grammes de caractères (2 à 5, sensibles aux marques morphologiques et lexicales régionales) +
régression logistique équilibrée.

**Protocole.** Validation croisée **groupée par talk** (GroupKFold) : un talk n'est jamais à la fois en
entraînement et en test. Pour montrer pourquoi c'est indispensable, on refait l'expérience avec un découpage
aléatoire des segments (fuite : mêmes locuteurs et mêmes sujets des deux côtés).

In [ ]:
def cv_eval(X, y, groups, splitter, make_clf):
    preds = np.empty(len(y), dtype=object)
    for tr, te in splitter.split(X, y, groups):
        clf = make_clf(); clf.fit([X[i] for i in tr] if isinstance(X, list) else X[tr], y[tr])
        preds[te] = clf.predict([X[i] for i in te] if isinstance(X, list) else X[te])
    p, r, f, _ = precision_recall_fscore_support(y, preds, average="macro", zero_division=0)
    return preds, {"accuracy": accuracy_score(y, preds), "précision macro": p, "rappel macro": r, "F1 macro": f}

text_clf = lambda: make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=200_000),
                                 LogisticRegression(max_iter=2000, class_weight="balanced", C=2.0))
results = []
if FEASIBLE:
    X = D.transcription.tolist(); y = D.region.values; g = D.talk.values
    n_folds = min(5, D.groupby("region").talk.nunique().min())
    maj = pd.Series(y).value_counts().index[0]
    p_, r_, f_, _ = precision_recall_fscore_support(y, [maj] * len(y), average="macro", zero_division=0)
    results.append({"tâche": "région (texte)", "protocole": "classe majoritaire", "accuracy": np.mean(y == maj), "précision macro": p_, "rappel macro": r_, "F1 macro": f_})
    pred_g, m_g = cv_eval(X, y, g, GroupKFold(n_folds), text_clf)
    results.append({"tâche": "région (texte)", "protocole": f"GroupKFold par talk ({n_folds} plis)", **m_g})
    _, m_r = cv_eval(X, y, g, KFold(5, shuffle=True, random_state=SEED), text_clf)
    results.append({"tâche": "région (texte)", "protocole": "KFold aléatoire (FUITE, pour comparaison)", **m_r})
    # décision au niveau du talk : vote majoritaire des segments
    talk_pred = pd.DataFrame({"talk": g, "y": y, "p": pred_g}).groupby("talk").agg(y=("y", "first"), p=("p", lambda s: s.value_counts().index[0]))
    results.append({"tâche": "région (texte)", "protocole": "GroupKFold + vote par talk", "accuracy": (talk_pred.y == talk_pred.p).mean(),
                    **dict(zip(["précision macro", "rappel macro", "F1 macro"], precision_recall_fscore_support(talk_pred.y, talk_pred.p, average="macro", zero_division=0)[:3]))})
    display(pd.DataFrame(results).round(3))
    print(classification_report(y, pred_g, zero_division=0))
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ConfusionMatrixDisplay(confusion_matrix(y, pred_g, labels=keep, normalize="true"), display_labels=keep).plot(ax=ax, cmap="Oranges", values_format=".2f", colorbar=False)
    ax.set_title("Région (texte), GroupKFold, normalisé par ligne"); plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "04_confusion_region_text.png", bbox_inches="tight"); plt.show()

**Lecture attendue.** Si l'accuracy en KFold aléatoire est très supérieure à celle du GroupKFold, le modèle
apprend surtout à reconnaître les talks (sujet, vocabulaire de l'orateur), pas une variante régionale.
Les n-grammes les plus discriminants (cellule suivante) permettent de vérifier s'il s'agit de marqueurs
dialectaux plausibles ou de mots liés au sujet des talks.

In [ ]:
if FEASIBLE:
    clf = text_clf().fit(X, y)
    vec, lr = clf.steps[0][1], clf.steps[1][1]
    names = np.array(vec.get_feature_names_out())
    top = {c: ", ".join(repr(n.strip()) for n in names[np.argsort(lr.coef_[i])[-12:][::-1]]) for i, c in enumerate(lr.classes_)}
    display(pd.Series(top, name="n-grammes les plus associés").to_frame())

## 5. Classifieurs audio
Représentation : moyenne temporelle des états cachés de la couche 12 de **w2v-BERT-2.0** (encodeur pré-entraîné,
non fine-tuné), qui capte l'accent et la prosodie autant que le contenu. Classifieur : régression logistique.

- **5.1 Région (audio)** : même protocole groupé par talk ; au plus 40 segments par talk.
- **5.2 Type de code-switching (audio)** : TUN / MIXED / FOR, étiquettes dérivées des transcriptions (fiables) ;
  splits officiels train / test (talks disjoints).

In [ ]:
def embed(paths, layer=12, bs=16):
    from transformers import AutoFeatureExtractor, Wav2Vec2BertModel
    fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")
    m = Wav2Vec2BertModel.from_pretrained("facebook/w2v-bert-2.0", torch_dtype=torch.float16).cuda().eval()
    out = []
    for i in range(0, len(paths), bs):
        w = [data.load_audio(p)[: 16000 * 15] for p in paths[i:i + bs]]
        f = fe(w, sampling_rate=16000, return_tensors="pt", padding=True, return_attention_mask=True)
        with torch.no_grad():
            h = m(f["input_features"].cuda().half(), attention_mask=f["attention_mask"].cuda(), output_hidden_states=True).hidden_states[layer]
        mask = f["attention_mask"].cuda().unsqueeze(-1).half()
        out.append(((h * mask).sum(1) / mask.sum(1)).float().cpu().numpy())
    del m; torch.cuda.empty_cache()
    return np.concatenate(out)

audio_clf = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5))
if RUN_AUDIO:
    Dall = data.cut_segments(df)
    if FEASIBLE:
        Da = Dall[Dall.region.isin(keep)].groupby("talk", group_keys=False).apply(lambda x: x.sample(min(40, len(x)), random_state=SEED)).reset_index(drop=True)
        Ea = embed(Da.path.tolist())
        n_folds = min(5, Da.groupby("region").talk.nunique().min())
        pred_a, m_a = cv_eval(Ea, Da.region.values, Da.talk.values, GroupKFold(n_folds), audio_clf)
        results.append({"tâche": "région (audio)", "protocole": f"GroupKFold par talk ({n_folds} plis)", **m_a})
    tr = Dall[Dall.split == "train"].groupby("seg_type", group_keys=False).apply(lambda x: x.sample(min(1500, len(x)), random_state=SEED))
    te = Dall[Dall.split == "test"]
    if SMOKE:   # essai rapide : pseudo-test = un talk de train tenu à part (le test officiel n'est pas utilisé)
        _t = sorted(Dall[Dall.split == "train"].talk.unique())[:5]
        te = Dall[Dall.talk == _t[4]]
        tr = Dall[Dall.talk.isin(_t[:4])].groupby("seg_type", group_keys=False).apply(lambda x: x.sample(min(60, len(x)), random_state=SEED))
    Etr, Ete = embed(tr.path.tolist()), embed(te.path.tolist())
    clf_cs = audio_clf().fit(Etr, tr.seg_type.values)
    pc = clf_cs.predict(Ete)
    p_, r_, f_, _ = precision_recall_fscore_support(te.seg_type, pc, average="macro", zero_division=0)
    results.append({"tâche": "type de code-switching (audio)", "protocole": "train → test officiel", "accuracy": accuracy_score(te.seg_type, pc),
                    "précision macro": p_, "rappel macro": r_, "F1 macro": f_})
    print(classification_report(te.seg_type, pc, zero_division=0))
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(te.seg_type, pc, labels=["TUN", "MIXED", "FOR"], normalize="true"), display_labels=["TUN", "MIXED", "FOR"]).plot(ax=ax, cmap="Greens", values_format=".2f", colorbar=False)
    ax.set_title("Type de segment depuis l'audio (test)"); plt.tight_layout(); plt.savefig(RESULTS_DIR / "04_confusion_cs_audio.png", bbox_inches="tight"); plt.show()

In [ ]:
res = pd.DataFrame(results).round(3)
if len(res):
    res.to_csv(RESULTS_DIR / "dialect_id_results.csv", index=False)
display(res)

## 6. Analyse critique et limites
- **Pas d'étiquettes dialectales** : la région de l'événement est un indicateur indirect et bruité. Un score
  élevé ne prouve pas qu'une variante dialectale est reconnue ; un score faible ne prouve pas l'absence de variation.
- **Confusion entre locuteur, sujet et région** : avec quelques talks par région, chaque région est représentée
  par une poignée d'orateurs. L'écart GroupKFold / KFold aléatoire quantifie cette confusion.
- **Déséquilibre** : le Grand Tunis domine probablement ; les métriques macro et le `class_weight` compensent
  partiellement, la matrice de confusion montre vers quelles classes se fait l'attraction.
- **Parole préparée** : en contexte TEDx, les orateurs tendent vers un tunisien « central » (koïné de Tunis),
  ce qui atténue les traits régionaux.
- **Tâche code-switching** : étiquettes fiables mais dérivées automatiquement du script des mots ; un mot
  français transcrit en arabe sera compté comme tunisien.

**Pour aller plus loin** : collecter une annotation de l'origine des locuteurs (questionnaire), utiliser un
corpus avec métadonnées régionales (par ex. corpus parlés régionaux tunisiens) et évaluer le transfert ; utiliser
des marqueurs dialectaux connus (réalisation du qaf /q/ vs /g/, pronoms ena/na, négation) comme traits explicites.

## Références
- F. Bougares et al. *TEDxTN*, ArabicNLP 2025 (section « Ethical considerations and limitations »).
- H. Bouamor et al. *The MADAR Arabic Dialect Corpus and Lexicon*, LREC 2018.
- M. Abdul-Mageed et al. *NADI: Nuanced Arabic Dialect Identification shared tasks*, 2020-2023.
- Seamless Communication. *w2v-BERT 2.0*, 2023. F. Pedregosa et al. *Scikit-learn*, JMLR 2011.